# Segmentacion PASCAL VOC 2012 — Traditional vs Hyperbolic vs Pontryagin (U-Net)

Comparacion de los 4 heads de segmentacion sobre un benchmark publico multiclase (21 clases),
generalizando mas alla del UAV sugarcane binario privado usado en el paper:

  - `vanilla`     — U-Net + CE estandar (sin pesos de clase)
  - `euclidean`   — U-Net + CE ponderada por clase   (baseline "tradicional")
  - `hyperbolic`  — U-Net + Poincare-ball MLR
  - `pontryagin`  — U-Net + PRFE (J-hyperplane MLR)   — la propuesta de este trabajo

Misma arquitectura para los 4 (`run_seg_pascalvoc.py`, calcado de `run_seg_uav_unet.py`,
donde el paper reporta mIoU 0.878 para pontryagin sobre UAV sugarcane binario) corrida sobre
PASCAL VOC 2012.

```
0. GPU check
1. Mount Google Drive
2. Configuracion  <- EDITAR paths aqui
3. Clone repo + instalar deps
4. sys.path + import check
5. Dataset (descarga VOC2012 la primera vez, ~2GB)
6. HPO  (opcional, ~20-40 min por modelo en T4) — vanilla / euclidean / hyperbolic / pontryagin
7. Entrenamiento completo  (carga best_params.json automaticamente si existe) — los 4 modelos
8. Analisis y resultados  (guardado en Drive) — tabla + graficos comparando los 4
9. Descargar
```

> **Solo la celda 2 necesita edicion antes de correr.**
> VOID_LABEL=255 (bordes sin etiquetar de VOC) se excluye de la loss y de las metricas de IoU/Dice automaticamente.

## 0 · GPU check

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU — Runtime → Change runtime type → GPU')

## 1 · Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    print('Not running on Colab — Drive mount skipped.')

## 2 · Configuracion  ← EDIT THIS CELL

Set your Drive paths here.
`DATA_ROOT` cachea el VOCdevkit descargado (~2GB) para no repetir la descarga cada sesion.
Everything else runs automatically.

In [ ]:
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL           = 'https://github.com/craljimenez/rr-pontryagin-embedded.git'
    REPO_DIR           = '/content/rr-pontryagin-embedded'
    # VOC2012 has ~17k small files — extracting them onto a Drive-mounted
    # (FUSE) path is very slow. Keep the dataset on local disk; it's public
    # and re-downloads in a couple of minutes on a fresh runtime anyway.
    # Results (checkpoints, metrics, figures) DO go to Drive, so those
    # persist across sessions/disconnects.
    DATA_ROOT          = '/content/pascal_voc_data'
    DRIVE_RESULTS_ROOT = '/content/drive/MyDrive/seg_pascalvoc_results'
else:
    # Walk UP from CWD until pyproject.toml is found — immune to kernel CWD quirks.
    _cwd = Path(os.getcwd()).resolve()
    _repo = next(
        (p for p in [_cwd, *_cwd.parents] if (p / 'pyproject.toml').exists()),
        None,
    )
    assert _repo is not None, f'pyproject.toml not found above {_cwd}'
    REPO_DIR           = str(_repo)
    DATA_ROOT          = str(_repo / 'experiments' / 'data' / 'pascal_voc')
    DRIVE_RESULTS_ROOT = str(_repo / 'experiments' / 'results' / 'seg_pascalvoc')
    REPO_URL           = ''

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(DRIVE_RESULTS_ROOT, exist_ok=True)
print(f'IN_COLAB           : {IN_COLAB}')
print(f'REPO_DIR           : {REPO_DIR}')
print(f'DATA_ROOT          : {DATA_ROOT}')
print(f'DRIVE_RESULTS_ROOT : {DRIVE_RESULTS_ROOT}')
print('Config OK.')

## 3 · Clone repo + instalar dependencias

In [ ]:
import subprocess, sys
from pathlib import Path

if IN_COLAB:
    if not Path(REPO_DIR).exists():
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=True)
    print(f'Repo at: {REPO_DIR}')
else:
    print(f'Local mode — repo already at {REPO_DIR}')
print('Done.')

In [ ]:
import importlib.util, subprocess, sys, os
from pathlib import Path

# REPO_DIR was already cloned/verified two cells ago — trust it directly
# instead of walking up from CWD (CWD is still /content here, and the repo
# lives in a *subdirectory* of it, not an ancestor, so a walk-up never finds it).
_repo = Path(REPO_DIR).resolve()
assert (_repo / 'pyproject.toml').exists(), (
    f'pyproject.toml not found in {_repo} — did the clone in the previous cell succeed?'
)

def _ensure(import_name, *pip_args):
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {" ".join(str(a) for a in pip_args)} ...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', *pip_args, '-q'],
            check=True,
        )
        print(f'  {import_name} installed.')
    else:
        print(f'  {import_name} already installed — skipping.')

_ensure('prfe',        '-e', str(_repo))   # editable install
_ensure('skopt',       'scikit-optimize')
_ensure('PIL',         'Pillow')
_ensure('torchvision', 'torchvision')
_ensure('timm',        'timm')
_ensure('pandas',      'pandas')
print(f'Repo root : {_repo}')
print('Dependencies ready.')

## 4 · sys.path + import check

In [ ]:
import sys, os, importlib, glob
from pathlib import Path

# Trust REPO_DIR from the config cell instead of re-deriving it by walking
# up from CWD (same reasoning as the previous cell — CWD is not inside the
# repo yet at this point, so a walk-up search would never find src/prfe).
_repo = Path(REPO_DIR).resolve()
assert (_repo / 'src' / 'prfe').is_dir(), f'src/prfe/ not found in {_repo}'
REPO_DIR = str(_repo)

_exp = str(_repo / 'experiments')
_src = str(_repo / 'src')
_venv_sp = sorted(glob.glob(str(_repo / '.venv' / 'lib' / 'python*' / 'site-packages')))
_venv_sp = _venv_sp[-1] if _venv_sp else None

# 1. Remove stale/relative entries first
sys.path = [
    p for p in sys.path
    if Path(p).is_absolute()
    and not any(k in p for k in ['rr-pontryagin', '003_pontriagyn', 'embebbed'])
]

# 2. Re-add the paths we need (venv must come AFTER cleanup, not before)
for p in filter(None, [_venv_sp, _exp, _src]):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(_exp)
importlib.invalidate_caches()

print(f'REPO_DIR : {REPO_DIR}')
print(f'CWD      : {os.getcwd()}')
print(f'venv sp  : {_venv_sp}')

from configs.seg_pascalvoc import AVAILABLE_MODELS, N_CLASSES, CLASS_NAMES
print('Imports OK. Available models:', AVAILABLE_MODELS)
print(f'N_CLASSES: {N_CLASSES}')

## 5 · Dataset

Descarga PASCAL VOC 2012 (train=1464 / val=1449 imgs) via `torchvision.datasets.VOCSegmentation`
en `DATA_ROOT` la primera vez (~2GB, unos minutos). El split oficial "val" se divide
deterministicamente 50/50 en val/test (VOC no publica mascaras de test).

In [ ]:
from run_seg_pascalvoc import build_dataloaders

data = build_dataloaders(data_root=DATA_ROOT)
print(f"Train: {len(data['train_ds'])} | Val: {len(data['val_ds'])} | Test: {len(data['test_ds'])}")
img, mask = data['train_ds'][0]
print('image tensor:', img.shape, img.dtype)
print('mask  tensor:', mask.shape, mask.dtype, 'classes present:', sorted(mask.unique().tolist()))

## 6 · HPO  *(opcional — ~20-40 min por modelo en T4)*

Optimizacion bayesiana con `scikit-optimize` sobre mIoU. Salta automaticamente si
`best_params.json` ya existe en Drive. Puedes saltar esta seccion entera para entrenar
con los hiperparametros por defecto del config (heredados de la caña de azucar).

In [ ]:
import json
from pathlib import Path

for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'hpo' / 'best_params.json'
    if p.exists():
        d     = json.loads(p.read_text())
        score = d.get('_best_val_macro_iou', float('nan'))
        print(f'  {mt:<12}  HPO done  —  best val mIoU = {score:.4f}')
    else:
        print(f'  {mt:<12}  no HPO results yet')

### 6a · Vanilla

In [ ]:
if not (Path(DRIVE_RESULTS_ROOT) / 'vanilla' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_seg_pascalvoc.py \
        --model        vanilla \
        --n-calls      20 \
        --n-random     8 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('vanilla HPO already done — skipping.')

### 6b · Euclidean

In [ ]:
if not (Path(DRIVE_RESULTS_ROOT) / 'euclidean' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_seg_pascalvoc.py \
        --model        euclidean \
        --n-calls      20 \
        --n-random     8 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('euclidean HPO already done — skipping.')

### 6c · Hyperbolic

In [ ]:
if not (Path(DRIVE_RESULTS_ROOT) / 'hyperbolic' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_seg_pascalvoc.py \
        --model        hyperbolic \
        --n-calls      25 \
        --n-random     8 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('hyperbolic HPO already done — skipping.')

### 6d · Pontryagin

In [ ]:
if not (Path(DRIVE_RESULTS_ROOT) / 'pontryagin' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_seg_pascalvoc.py \
        --model        pontryagin \
        --n-calls      30 \
        --n-random     10 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('pontryagin HPO already done — skipping.')

## 7 · Entrenamiento completo

Carga `best_params.json` automaticamente si el HPO fue ejecutado (busca en
`<results>/<model>/hpo/best_params.json`); si no, usa los valores por defecto del config.
`EPOCHS`/`EARLY_STOP_PATIENCE` vienen de `configs/seg_pascalvoc.py` — pasa `--epochs` para
sobreescribir puntualmente.

### 7a · Vanilla

In [ ]:
!python {REPO_DIR}/experiments/run_seg_pascalvoc.py \
    --model       vanilla \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

### 7b · Euclidean

In [ ]:
!python {REPO_DIR}/experiments/run_seg_pascalvoc.py \
    --model       euclidean \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

### 7c · Hyperbolic

In [ ]:
!python {REPO_DIR}/experiments/run_seg_pascalvoc.py \
    --model       hyperbolic \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

### 7d · Pontryagin

In [ ]:
!python {REPO_DIR}/experiments/run_seg_pascalvoc.py \
    --model       pontryagin \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

## 8 · Analisis y resultados

### 8a · Tabla comparativa de metricas (test)

In [ ]:
import json
import pandas as pd
from pathlib import Path

rows = []
for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'metrics.json'
    if not p.exists():
        print(f'  {mt}: metrics.json not found — skip (entrena primero)'); continue
    d = json.loads(p.read_text())
    rows.append({
        'model':             mt,
        'best_val_macro_iou': round(d['best_val_macro_iou'], 4),
        'test_macro_iou':     round(d['macro_iou'], 4),
        'test_macro_dice':    round(d['macro_dice'], 4),
        'test_pixel_acc':     round(d['pixel_acc'], 4),
        'test_micro_iou':     round(d['micro_iou'], 4),
        'n_valid_classes':    d['n_valid_classes'],
    })

df = pd.DataFrame(rows).sort_values('test_macro_iou', ascending=False)
df

### 8b · IoU por clase — comparacion entre modelos

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

per_class = {}
for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'metrics_per_class.csv'
    if p.exists():
        per_class[mt] = pd.read_csv(p).set_index('class')['iou']

if per_class:
    pc_df = pd.DataFrame(per_class)
    pc_df = pc_df[pc_df.sum(axis=1) > 0]   # solo clases presentes en el test set
    ax = pc_df.plot(kind='bar', figsize=(14, 5), width=0.8)
    ax.set_ylabel('IoU')
    ax.set_title('IoU por clase — PASCAL VOC 2012 test')
    ax.legend(title='modelo')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No hay metrics_per_class.csv todavia — entrena al menos un modelo primero.')

### 8c · Paneles de prediccion (test)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'viz' / 'test_final.png'
    if p.exists():
        print(f'--- {mt} ---')
        display(Image(filename=str(p)))
    else:
        print(f'  {mt}: sin visualizacion todavia.')

## 9 · Descargar / explorar resultados

La copia en Drive ya es persistente. Esta celda permite descargar un zip a tu maquina local.

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    from pathlib import Path
    zip_path = '/content/seg_pascalvoc_results'
    shutil.make_archive(zip_path, 'zip', DRIVE_RESULTS_ROOT)
    size_mb = Path(zip_path + '.zip').stat().st_size / 1e6
    print(f'Archive: {zip_path}.zip  ({size_mb:.1f} MB)')
    files.download(zip_path + '.zip')
else:
    print(f'Local mode — results are at:\n  {DRIVE_RESULTS_ROOT}')